In [1]:
library(progress)
install.packages("dbarts")
install.packages("stochtree")
library(stochtree)
library(dbarts)
install.packages("mvtnorm")
library(mvtnorm)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘BH’



Attaching package: ‘dbarts’


The following object is masked from ‘package:stochtree’:

    bart


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



# DGP_1

In [2]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

num_gfr<-40

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
new_colnames <- c(
  # PEHE metrics
  "bcf_1k_pehe1", "bcf_1k_pehe2",
  "bcf_0.5k_pehe1", "bcf_0.5k_pehe2",
  "bcf_0.25k_pehe1", "bcf_0.25k_pehe2",
  "bcf_0.1k_pehe1", "bcf_0.1k_pehe2",
  "bcf_0.05k_pehe1", "bcf_0.05k_pehe2",

  # RMSE metrics (added)
  "bcf_1k_rmse1", "bcf_1k_rmse2",
  "bcf_0.5k_rmse1", "bcf_0.5k_rmse2",
  "bcf_0.25k_rmse1", "bcf_0.25k_rmse2",
  "bcf_0.1k_rmse1", "bcf_0.1k_rmse2",
  "bcf_0.05k_rmse1", "bcf_0.05k_rmse2",

  # MAPE metrics (added)
  "bcf_1k_mape1", "bcf_1k_mape2",
  "bcf_0.5k_mape1", "bcf_0.5k_mape2",
  "bcf_0.25k_mape1", "bcf_0.25k_mape2",
  "bcf_0.1k_mape1", "bcf_0.1k_mape2",
  "bcf_0.05k_mape1", "bcf_0.05k_mape2",

  # Tau 95% interval width metrics
  "bcf_1k_tau_951", "bcf_1k_tau_952",
  "bcf_0.5k_tau_951", "bcf_0.5k_tau_952",
  "bcf_0.25k_tau_951", "bcf_0.25k_tau_952",
  "bcf_0.1k_tau_951", "bcf_0.1k_tau_952",
  "bcf_0.05k_tau_951", "bcf_0.05k_tau_952",

  # Tau 95% interval width metrics (weighted)
  "bcf_1k_tau_951w", "bcf_1k_tau_952w",
  "bcf_0.5k_tau_951w", "bcf_0.5k_tau_952w",
  "bcf_0.25k_tau_951w", "bcf_0.25k_tau_952w",
  "bcf_0.1k_tau_951w", "bcf_0.1k_tau_952w",
  "bcf_0.05k_tau_951w", "bcf_0.05k_tau_952w"
)

# Calculate the total number of columns
num_columns <- length(new_colnames)

# Initialize the results matrix with the correct number of columns
results_matrix <- matrix(NA, nrow = num_simulations, ncol = num_columns)
colnames(results_matrix) <- new_colnames

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-rnorm(n)
X2<-rnorm(n)
X3<-rnorm(n)
X4<-rbinom(n, size = 1, prob = 0.5)
X5<-sample(1:3, size = n, replace = TRUE)

g_values <- c(2, -1, -4) # g(1)=2, g(2)=-1, g(3)=-4
g_x5 <- g_values[X5]

X<-cbind(X1, X2, X3, X4, X5)

Mu1<- -6 + g_x5 + 6 * abs(X3 - 1)+ X1 * X3
Mu2<- -4 + 0.66*g_x5 + 8.5 * abs(X3 - 0.85)+ 0.25*X1 * X3

Tau1<- 1 + 2 * X2 * X4
Tau2<- -1 + 3.5 * X2 * X4

s_linear <- sd(Mu1)

pi <- 0.8 * pnorm(3 * Mu1 / s_linear - 0.5 * X1) + 0.05 + runif(n) / 10

true_propensity<-pmin(1, pmax(0, pi))

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(1, 0, 0, 1), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-rnorm(n_test)
X2_test<-rnorm(n_test)
X3_test<-rnorm(n_test)
X4_test<-rbinom(n_test, size = 1, prob = 0.5)
X5_test<-sample(1:3, size = n_test, replace = TRUE)

g_x5_test <- g_values[X5_test]

X_test<-cbind(X1_test, X2_test, X3_test, X4_test, X5_test)

Mu1_test<- -6 + g_x5_test + 6 * abs(X3_test - 1)+ X1_test * X3_test
Mu2_test<- -4 + 0.66*g_x5_test + 8.5 * abs(X3_test - 0.85)+ 0.25*X1_test * X3_test

Tau1_test<- 1 + 2 * X2_test * X4_test
Tau2_test<- -1 + 3.5 * X2_test * X4_test

s_linear_test <- sd(Mu1_test)

pi_test <- 0.8 * pnorm(3 * Mu1_test / s_linear_test - 0.5 * X1_test) + 0.05 + runif(n_test) / 10

true_propensity_test<-pmin(1, pmax(0, pi_test))

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(1, 0, 0, 1), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

num_gfr<-40
n_iter<-25

bcf_1k_mod1 <- bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))


bcf_1k_mod2 <- bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))

bcf_1k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_1k_mod1$tau_hat_test))^2))

bcf_1k_tau_951<-mean(diag(apply(bcf_1k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))

bcf_1k_tau_951w<-mean(apply(bcf_1k_mod1$tau_hat_test, 1, cred_width, 0.95))

bcf_1k_rmse1 <- sqrt(mean((Y_test[,1] - bcf_1k_mod1$y_hat_test)^2))
bcf_1k_mape1 <- mean((abs(Y_test[,1] - bcf_1k_mod1$y_hat_test))/abs(Y_test[,1]))

bcf_1k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_1k_mod2$tau_hat_test))^2))

bcf_1k_tau_952<-mean(diag(apply(bcf_1k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_1k_tau_952w<-mean(apply(bcf_1k_mod2$tau_hat_test, 1, cred_width, 0.95))

bcf_1k_rmse2 <- sqrt(mean((Y_test[,2] - bcf_1k_mod2$y_hat_test)^2))
bcf_1k_mape2 <- mean((abs(Y_test[,2] - bcf_1k_mod2$y_hat_test))/abs(Y_test[,2]))

n_iter<-13

bcf_0.5k_mod1 <- bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))


bcf_0.5k_mod2 <- bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))

bcf_0.5k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.5k_mod1$tau_hat_test))^2))

bcf_0.5k_tau_951<-mean(diag(apply(bcf_0.5k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))

bcf_0.5k_tau_951w<-mean(apply(bcf_0.5k_mod1$tau_hat_test, 1, cred_width, 0.95))

bcf_0.5k_rmse1 <- sqrt(mean((Y_test[,1] - bcf_0.5k_mod1$y_hat_test)^2))
bcf_0.5k_mape1 <- mean((abs(Y_test[,1] - bcf_0.5k_mod1$y_hat_test))/abs(Y_test[,1]))

bcf_0.5k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.5k_mod2$tau_hat_test))^2))

bcf_0.5k_tau_952<-mean(diag(apply(bcf_0.5k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.5k_tau_952w<-mean(apply(bcf_0.5k_mod2$tau_hat_test, 1, cred_width, 0.95))

bcf_0.5k_rmse2 <- sqrt(mean((Y_test[,2] - bcf_0.5k_mod2$y_hat_test)^2))
bcf_0.5k_mape2 <- mean((abs(Y_test[,2] - bcf_0.5k_mod2$y_hat_test))/abs(Y_test[,2]))

n_iter<-6

bcf_0.25k_mod1 <- bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))


bcf_0.25k_mod2 <- bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))

bcf_0.25k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.25k_mod1$tau_hat_test))^2))

bcf_0.25k_tau_951<-mean(diag(apply(bcf_0.25k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))

bcf_0.25k_tau_951w<-mean(apply(bcf_0.25k_mod1$tau_hat_test, 1, cred_width, 0.95))

bcf_0.25k_rmse1 <- sqrt(mean((Y_test[,1] - bcf_0.25k_mod1$y_hat_test)^2))
bcf_0.25k_mape1 <- mean((abs(Y_test[,1] - bcf_0.25k_mod1$y_hat_test))/abs(Y_test[,1]))

bcf_0.25k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.25k_mod2$tau_hat_test))^2))

bcf_0.25k_tau_952<-mean(diag(apply(bcf_0.25k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.25k_tau_952w<-mean(apply(bcf_0.25k_mod2$tau_hat_test, 1, cred_width, 0.95))

bcf_0.25k_rmse2 <- sqrt(mean((Y_test[,2] - bcf_0.25k_mod2$y_hat_test)^2))
bcf_0.25k_mape2 <- mean((abs(Y_test[,2] - bcf_0.25k_mod2$y_hat_test))/abs(Y_test[,2]))

n_iter<-3

bcf_0.1k_mod1 <- bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))


bcf_0.1k_mod2 <- bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))

bcf_0.1k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.1k_mod1$tau_hat_test))^2))

bcf_0.1k_tau_951<-mean(diag(apply(bcf_0.1k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))

bcf_0.1k_tau_951w<-mean(apply(bcf_0.1k_mod1$tau_hat_test, 1, cred_width, 0.95))

bcf_0.1k_rmse1 <- sqrt(mean((Y_test[,1] - bcf_0.1k_mod1$y_hat_test)^2))
bcf_0.1k_mape1 <- mean((abs(Y_test[,1] - bcf_0.1k_mod1$y_hat_test))/abs(Y_test[,1]))

bcf_0.1k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.1k_mod2$tau_hat_test))^2))

bcf_0.1k_tau_952<-mean(diag(apply(bcf_0.1k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.1k_tau_952w<-mean(apply(bcf_0.1k_mod2$tau_hat_test, 1, cred_width, 0.95))

bcf_0.1k_rmse2 <- sqrt(mean((Y_test[,2] - bcf_0.1k_mod2$y_hat_test)^2))
bcf_0.1k_mape2 <- mean((abs(Y_test[,2] - bcf_0.1k_mod2$y_hat_test))/abs(Y_test[,2]))

n_iter<-40

bcf_0.05k_mod1 <- bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))


bcf_0.05k_mod2 <- bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 0, general_params = list(num_chains = num_gfr))

bcf_0.05k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.05k_mod1$tau_hat_test))^2))

bcf_0.05k_tau_951<-mean(diag(apply(bcf_0.05k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))

bcf_0.05k_tau_951w<-mean(apply(bcf_0.05k_mod1$tau_hat_test, 1, cred_width, 0.95))

bcf_0.05k_rmse1 <- sqrt(mean((Y_test[,1] - bcf_0.05k_mod1$y_hat_test)^2))
bcf_0.05k_mape1 <- mean((abs(Y_test[,1] - bcf_0.05k_mod1$y_hat_test))/abs(Y_test[,1]))

bcf_0.05k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.05k_mod2$tau_hat_test))^2))

bcf_0.05k_tau_952<-mean(diag(apply(bcf_0.05k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.05k_tau_952w<-mean(apply(bcf_0.05k_mod2$tau_hat_test, 1, cred_width, 0.95))

bcf_0.05k_rmse2 <- sqrt(mean((Y_test[,2] - bcf_0.05k_mod2$y_hat_test)^2))
bcf_0.05k_mape2 <- mean((abs(Y_test[,2] - bcf_0.05k_mod2$y_hat_test))/abs(Y_test[,2]))

# Store the results in the matrix
results_matrix[i, ] <- c(
  # PEHE metrics
  bcf_1k_pehe1, bcf_1k_pehe2,
  bcf_0.5k_pehe1, bcf_0.5k_pehe2,
  bcf_0.25k_pehe1, bcf_0.25k_pehe2,
  bcf_0.1k_pehe1, bcf_0.1k_pehe2,
  bcf_0.05k_pehe1, bcf_0.05k_pehe2,

  # RMSE metrics
  bcf_1k_rmse1, bcf_1k_rmse2,
  bcf_0.5k_rmse1, bcf_0.5k_rmse2,
  bcf_0.25k_rmse1, bcf_0.25k_rmse2,
  bcf_0.1k_rmse1, bcf_0.1k_rmse2,
  bcf_0.05k_rmse1, bcf_0.05k_rmse2,

  # MAPE metrics
  bcf_1k_mape1, bcf_1k_mape2,
  bcf_0.5k_mape1, bcf_0.5k_mape2,
  bcf_0.25k_mape1, bcf_0.25k_mape2,
  bcf_0.1k_mape1, bcf_0.1k_mape2,
  bcf_0.05k_mape1, bcf_0.05k_mape2,

  # Tau 95% interval width metrics
  bcf_1k_tau_951, bcf_1k_tau_952,
  bcf_0.5k_tau_951, bcf_0.5k_tau_952,
  bcf_0.25k_tau_951, bcf_0.25k_tau_952,
  bcf_0.1k_tau_951, bcf_0.1k_tau_952,
  bcf_0.05k_tau_951, bcf_0.05k_tau_952,

  # Tau 95% interval width metrics (weighted)
  bcf_1k_tau_951w, bcf_1k_tau_952w,
  bcf_0.5k_tau_951w, bcf_0.5k_tau_952w,
  bcf_0.25k_tau_951w, bcf_0.25k_tau_952w,
  bcf_0.1k_tau_951w, bcf_0.1k_tau_952w,
  bcf_0.05k_tau_951w, bcf_0.05k_tau_952w
)

cat("Iteration:", i)

}

# Export the results matrix to a CSV file
write.csv(results_matrix, "BCF_simulation_results_DGP1.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to BCF_simulation_results_DGP1.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position”


Iteration: 1

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position”


Iteration: 2

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position”


...

Iteration: 98

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position”


Iteration: 99

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position”


Iteration: 100Simulation completed and results saved to BCF_simulation_results_DGP1.csv


In [3]:
print(results_matrix)

       bcf_1k_pehe1 bcf_1k_pehe2 bcf_0.5k_pehe1 bcf_0.5k_pehe2 bcf_0.25k_pehe1
  [1,]    0.8669375    1.0536684      1.0334614       1.292393       1.1433799
  [2,]    0.9150573    1.0527451      0.9996607       1.309180       1.1108945
  [3,]    0.9088160    1.2658239      1.0250091       1.473728       1.2334511
  [4,]    0.7848927    1.0540179      0.9035467       1.189833       1.0204923
  [5,]    0.9235131    1.2157568      1.0646379       1.398173       1.3053147
  [6,]    0.9305471    0.8705149      0.9624054       1.190325       1.1006772
  [7,]    0.8291413    1.1416367      0.9871016       1.332099       1.0808869
  [8,]    0.8678992    1.2015594      0.9649045       1.486716       1.0638084
  [9,]    0.6777435    1.1170227      0.8431115       1.344417       0.9750702
 [10,]    1.1313793    1.1624998      1.2519147       1.411156       1.3413172
 [11,]    0.6865621    1.1141350      0.9015784       1.291063       0.9978560
 [12,]    0.7628959    0.9602954      0.9378481     